# A1: Data Sources Setup

---

### Overview - A Notebooks download and clean data, B notebooks create timelines. C notebooks analyze. D notebooks publish the City of Berkeley Daily Update: the Permit Pipeline, Inspection Pipeline, and Construction Pipeline.

## This set of notebooks complement and support the City of Berkeley Annual Update, to be submitted in March, 2026, to CA HCD per legal requirement.

This notebook connects to the Berkeley Open Data Portal API to downloads permit and inspection data.

## These notebooks are designed to extend the 2019 and 2023 Terner Center "Making It Pencil" series by David Garcia. The Terner Center created an illustrative website examining a small set of alternative options available to developers, but failed to present their calculations. Building computational notebooks for every potential project in Berkeley will extend the Terner Center's vision.

**Inputs:** None (fetches from API)

**Outputs:**
- `zoning_permits.csv`
- `building_permits.csv`
- `planning_records.csv`

**Dependencies:** sodapy, pandas

---

## Data Collection Methods

### ✅ API Access (Working)
- **Business Licenses** - API works!

### ❌ API Blocked (Manual Download Required)
- **Zoning Permits** - Download Excel from Accela
- **Building Permits** - Download CSV from Open Data Portal

### Why Manual Downloads?
Berkeley's Web Application Firewall blocks programmatic access to prevent server overload. This is a common practice in civic data and teaches students:
- Real-world API limitations
- Backup data collection strategies
- Manual workflow documentation
```

---

## 🎯 **Where to Get the Files:**

### **Zoning Permits (Excel):**
```
1. Go to: https://aca-prod.accela.com/BERKELEY/Cap/CapHome.aspx?module=Planning
2. Login may be required (or public access)
3. Search/filter for active projects
4. Export to Excel
5. Save as: inputs/Active_Zoning_Projects.xlsx
```

### **Building Permits (CSV):**
```
1. Go to: https://data.cityofberkeley.info/d/ydr8-5enu
2. Click: Export → CSV
3. Save as: inputs/building_permits_raw.csv

In [1]:
# ============================================================================
# COLAB ENVIRONMENT SETUP (Run this first in Colab!)
# ============================================================================

import os
import sys
from pathlib import Path

print('🔧 SETTING UP ENVIRONMENT')
print('='*70)

# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print('🌐 Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('💻 Running locally')

if IN_COLAB:
    # Clone repository
    repo_path = Path('/content/berkeley-housing-analysis')
    
    if not repo_path.exists():
        print('\n📥 Cloning repository...')
        !git clone https://github.com/blockXblock/berkeley-housing-analysis.git
        print('✅ Repository cloned')
    else:
        print('\n✅ Repository already exists')
        # Pull latest changes
        !cd /content/berkeley-housing-analysis && git pull
    
    # Change to repo directory
    os.chdir(repo_path)
    
    # Add to Python path for module imports
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))
    
    # Create directories
    (repo_path / 'inputs').mkdir(exist_ok=True)
    (repo_path / 'outputs').mkdir(exist_ok=True)
    (repo_path / 'temp').mkdir(exist_ok=True)
    
    # Set DATA_DIR for compatibility with notebook
    DATA_DIR = repo_path
    
    print(f'\n📂 Working directory: {os.getcwd()}')
    print(f'📦 Python path updated for module imports')
    
    # Verify modules
    modules_path = repo_path / 'modules'
    if modules_path.exists():
        py_files = [f.name for f in modules_path.glob('*.py') if f.name != '__pycache__']
        print(f'✅ Found {len(py_files)} module files:')
        for name in sorted(py_files):
            print(f'   • {name}')
    
    print('\n⚠️  Note: API token not available in Colab')
    print('   Berkeley API is blocked anyway (403 errors)')
    print('   Will use manual data download method')

else:
    # Local environment
    if 'workflows' in os.getcwd():
        # Navigate up to berkeley-data root
        while 'berkeley-data' not in os.path.basename(os.getcwd()) and os.getcwd() != '/':
            os.chdir('..')
            if os.path.basename(os.getcwd()) == 'berkeley-data':
                break
    
    DATA_DIR = Path.cwd()
    print(f'\n📂 Working directory: {os.getcwd()}')

print('\n' + '='*70)
print('🎉 SETUP COMPLETE! Ready to run notebook.')
print('='*70)


🔧 SETTING UP ENVIRONMENT
💻 Running locally

📂 Working directory: /Users/johngage/berkeley-data

🎉 SETUP COMPLETE! Ready to run notebook.


## 1. Setup & Imports

In [2]:
# 1
# Install sodapy
!pip install -q sodapy


# Portable config loading
import json
from pathlib import Path
import shutil

# Try different paths
for p in [Path('config/berkeley_config.json'), Path('../00_config/berkeley_config.json')]:
    if p.exists():
        config_path = p
        break
else:
    # Create from template
    t = Path('config/berkeley_config.json.template')
    c = Path('config/berkeley_config.json')
    if t.exists():
        shutil.copy(t, c)
        config_path = c

with open(config_path) as f:
    CONFIG = json.load(f)

from modules.data_loader import get_socrata_client, load_permits_from_api, DATASETS
import pandas as pd

DATA_DIR = Path('.')
print(f'Config: {config_path}')
print(f'Datasets: {list(DATASETS.keys())}')


Config: config/berkeley_config.json
Datasets: ['business_licenses', 'building_permits', 'zoning_permits', 'planning_records', 'crime_incidents', 'restaurant_inspections']


In [3]:
# ============================================================================
# WORKING SOCRATA CLIENT (Using App Token)
# ============================================================================

from sodapy import Socrata
import pandas as pd

# Berkeley Open Data Portal
BERKELEY_DOMAIN = 'data.cityofberkeley.info'

# Your app token (provides higher rate limits)
APP_TOKEN = '8PDke1Hu50Wk65wSM0QPxmH1w'

# Dataset IDs (verified working)
WORKING_DATASETS = {
    'business_licenses': 'rwnf-bu3w',
    'crime_incidents': 'k2nh-s5h5',
    'restaurant_inspections': 'b47j-kakm',
    'building_permits': 'ydr8-5enu',
}

# Initialize client
client = Socrata(BERKELEY_DOMAIN, APP_TOKEN)

print('✅ Connected to Berkeley Open Data Portal')
print(f'   Domain: {BERKELEY_DOMAIN}')
print(f'   Using app token: {APP_TOKEN[:8]}...')
print(f'   Available datasets: {list(WORKING_DATASETS.keys())}')
print('\n⚠️  Note: Some datasets may still return 403 due to WAF')


✅ Connected to Berkeley Open Data Portal
   Domain: data.cityofberkeley.info
   Using app token: 8PDke1Hu...
   Available datasets: ['business_licenses', 'crime_incidents', 'restaurant_inspections', 'building_permits']

⚠️  Note: Some datasets may still return 403 due to WAF


In [4]:
# ============================================================================
# DATA FETCHING FUNCTION
# ============================================================================

def fetch_berkeley_data(dataset_name, limit=10000, filters=None):
    """
    Fetch data from Berkeley Open Data Portal
    
    Parameters:
    -----------
    dataset_name : str
        Name of dataset from WORKING_DATASETS dict
    limit : int
        Maximum number of records to fetch
    filters : dict
        Optional filters (e.g., {'city': 'Berkeley'})
    
    Returns:
    --------
    pandas.DataFrame or None
    """
    try:
        dataset_id = WORKING_DATASETS.get(dataset_name)
        if not dataset_id:
            raise ValueError(f'Unknown dataset: {dataset_name}')
        
        print(f'📥 Fetching {dataset_name}...')
        print(f'   Dataset ID: {dataset_id}')
        print(f'   Limit: {limit:,} records')
        print(f'   APP_TOKEN: {APP_TOKEN}')

        # Build query parameters
        params = {'$limit': limit}
        
        if filters:
            # Convert filters to SoQL WHERE clause
            where_clauses = [f"{k}='{v}'" for k, v in filters.items()]
            params['$where'] = ' AND '.join(where_clauses)
            print(f'   Filters: {filters}')
        
        # Fetch data
        results = client.get(dataset_id, **params)
        
        # Convert to DataFrame
        df = pd.DataFrame.from_records(results)
        
        print(f'✅ Success! Fetched {len(df):,} records')
        print(f'   Columns: {list(df.columns)[:5]}...')
        
        return df
        
    except Exception as e:
        print(f'❌ Error: {str(e)}')
        
        if '403' in str(e) or 'Forbidden' in str(e):
            print('\n   This dataset is blocked by WAF.')
            print('   Use manual download instead:')
            print(f'   https://data.cityofberkeley.info/d/{dataset_id}')
        
        return None

print('✅ fetch_berkeley_data() function ready')


✅ fetch_berkeley_data() function ready


In [5]:
# ============================================================================
# UTILITY FUNCTIONS - Timestamps and Timing 2026Feb
# ============================================================================

from datetime import datetime
import time

def cell_timestamp(label=""):
    """Print timestamp for cell execution"""
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"🕐 {label}")
    print(f"   Executed: {now}")
    print("="*70)

def timer_start():
    """Start execution timer"""
    global _cell_start_time
    _cell_start_time = time.time()
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"🕐 Started: {now}")
    print("="*70)

def timer_end():
    """End execution timer and show duration"""
    duration = time.time() - _cell_start_time
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print("="*70)
    print(f"🕐 Completed: {now}")
    print(f"⏱️  Duration: {duration:.2f} seconds")
    
print("✅ Timestamp utilities loaded")

✅ Timestamp utilities loaded


In [8]:
# ============================================================================
# TEST: Fetch Business Licenses
# ============================================================================

cell_timestamp(" Business Licenses")

timer_start()

print('🧪 TESTING API ACCESS')
print('='*70)

# Try to fetch business licenses (most likely to work)
df_business = fetch_berkeley_data('business_licenses', limit=100)

if df_business is not None:
    print('\n📊 SAMPLE DATA:')
    print(df_business.head())
    
    print(f'\n📈 SUMMARY:')
    print(f'   Total records: {len(df_business):,}')
    print(f'   Columns: {len(df_business.columns)}')
    
    # Save to CSV
    output_file = DATA_DIR / 'business_licenses.csv'
    df_business.to_csv(output_file, index=False)
    print(f'\n💾 Saved to: {output_file}')
else:
    print('\n⚠️  API access blocked - use manual download')

print('='*70)

timer_end()



🕐  Business Licenses
   Executed: 2026-02-17 15:41:32
🕐 Started: 2026-02-17 15:41:32
🧪 TESTING API ACCESS
📥 Fetching business_licenses...
   Dataset ID: rwnf-bu3w
   Limit: 100 records
   APP_TOKEN: 8PDke1Hu50Wk65wSM0QPxmH1w
✅ Success! Fetched 100 records
   Columns: ['apn', 'recordid', 'busdesc', 'b1_per_sub_type', 'dba']...

📊 SAMPLE DATA:
             apn   recordid              busdesc  \
0  056 198302300  BL-022398  SIGN/LETTERING SHOP   
1  057 203001000  BL-022671                 DELI   
2  052 153100900  BL-049175  ARCHITECTURE OFFICE   
3  ZZZZZZZZZZZZZ  BL-013782     TELECOM SERVICES   
4  052 156300500  BL-050197            BOOKSTORE   

                 b1_per_sub_type                            dba  \
0  Professional SemiProfessional                   DEAN'S SIGNS   
1                   Retail Trade                  E Z STOP DELI   
2  Professional SemiProfessional  GHA/GEOFFREY HOLTON AND ASSOC   
3   Business Personal Repair Svs                         LIVELY   
4       

## 2. API Configuration

Get your free API token from:
https://data.cityofberkeley.info/profile/edit/developer_settings

In [9]:
# Load environment variables
try:
    from dotenv import load_dotenv
    load_dotenv(DATA_DIR / '.env')
    print("Loaded .env file")
except:
    print("Note: python-dotenv not installed (optional)")

# Check for API token
APP_TOKEN = os.environ.get('BERKELEY_APP_TOKEN')

# TODO: If no environment variable, set your token here:
APP_TOKEN = "z1ZX3Y2jwZ_BCAoo_iIe1h14HMMAzjpPOV_M"

if APP_TOKEN:
    print(f"API token loaded: {APP_TOKEN[:8]}...")
else:
    print("WARNING: No API token found!")
    print("Get your free token at: https://data.cityofberkeley.info/profile/edit/developer_settings")

Loaded .env file
API token loaded: z1ZX3Y2j...


## 3. Available Datasets

Berkeley Open Data Portal datasets relevant to housing:

In [10]:
# Display available datasets
print("Berkeley Open Data - Housing Related Datasets:\n")
print("="*60)

for name, dataset_id in DATASETS.items():
    info = CONFIG['api']['datasets'].get(name, {})
    desc = info.get('description', 'No description')
    print(f"{name}")
    print(f"  ID: {dataset_id}")
    print(f"  Description: {desc}")
    print()

Berkeley Open Data - Housing Related Datasets:

business_licenses
  ID: rwnf-bu3w
  Description: Active business licenses

building_permits
  ID: ydr8-5enu
  Description: Building permits

zoning_permits
  ID: vkhm-tsvp
  Description: Zoning permits

planning_records
  ID: rk4r-58ys
  Description: Planning records

crime_incidents
  ID: k2nh-s5h5
  Description: No description

restaurant_inspections
  ID: b47j-kakm
  Description: No description



## 4. Fetch Zoning Permits

Zoning permits are the first step in the housing development pipeline.

In [12]:
# Fetch zoning permits
print("Fetching Zoning Permits...")
print("="*60)

cell_timestamp("ZONING PERMITS - API")

timer_start()

df_zoning = load_permits_from_api(
    'zoning_permits',
    limit=50000,
    app_token=APP_TOKEN
)

if df_zoning is not None:
    print(f"\nShape: {df_zoning.shape}")
    print(f"\nColumns:")
    for col in df_zoning.columns:
        print(f"  - {col}")
 
    print(f"\nSample records:")
    display(df_zoning.head(3))
timer_end()


Fetching Zoning Permits...
🕐 ZONING PERMITS - API
   Executed: 2026-02-17 15:43:02
🕐 Started: 2026-02-17 15:43:02
Using app token: z1ZX3Y2j...
Fetching zoning_permits from Berkeley Open Data...
Error fetching data: 403 Client Error: Forbidden
🕐 Completed: 2026-02-17 15:43:02
⏱️  Duration: 0.34 seconds


In [13]:
# ============================================================================
# ALTERNATIVE: Import Zoning Permits from Accela Excel Export - Feb2026
# ============================================================================

print("📥 LOADING ZONING PERMITS FROM EXCEL")
print("="*70)

cell_timestamp("ZONING PERMITS - Manual Excel Import")


# Path to manually downloaded Excel file
# Must manually go to City of Berkeley website: 
#https://aca-prod.accela.com/BERKELEY/Cap/CapHome.aspx?module=Planning&TabName=Planning&TabList=Home%7C0%7CBuilding%7C1%7CHousing%7C2%7CPlanning%7C3%7CFire%7C4%7CLicenses%7C5%7CPublicWorks%7C6%7CCurrentTabIndex%7C3
# login to your account. go to Zoning; Tab at top: Reports; download .xlsx file, 1 of 6
# download to Google Drive; put path below
# https://docs.google.com/spreadsheets/d/1S6dGYohCEjZqXdLSsUXtvoA3w9jwz9qB?rtpof=true&usp=drive_fs
# https://docs.google.com/spreadsheets/d/1S6dGYohCEjZqXdLSsUXtvoA3w9jwz9qB/edit?usp=drive_link&ouid=111924636441288322199&rtpof=true&sd=true

excel_file = Path('https://docs.google.com/spreadsheets/d/1S6dGYohCEjZqXdLSsUXtvoA3w9jwz9qB/edit?usp=drive_link&ouid=111924636441288322199&rtpof=true&sd=true')

#excel_file = Path('inputs/Active_Zoning_Projects.xlsx')


if excel_file.exists():
    print(f"✅ Found Excel file: {excel_file}")

    timer_start()

    
    try:
        # Read Excel file
        df_zoning_excel = pd.read_excel(excel_file)
        
        print(f"   Loaded {len(df_zoning_excel)} zoning projects")
        print(f"   Columns: {list(df_zoning_excel.columns)[:5]}...")
        
        # Show sample
        print("\n📊 Sample data:")
        print(df_zoning_excel.head(3))
        
        # Store for later use
        df_zoning = df_zoning_excel
        
    except Exception as e:
        print(f"❌ Error reading Excel: {e}")
        df_zoning = None

    timer_end()

        
else:
    print(f"⚠️  Excel file not found: {excel_file}")
    print("\n📥 MANUAL DOWNLOAD REQUIRED:")
    print("   1. Go to: https://aca-prod.accela.com/BERKELEY/Cap/CapHome.aspx")
    print("      → Select 'Planning' module")
    print("   2. Export 'Active Zoning Projects' to Excel")
    print("   3. Save as: inputs/Active_Zoning_Projects.xlsx")
    print("   4. Re-run this cell")
    df_zoning = None

print("="*70)

📥 LOADING ZONING PERMITS FROM EXCEL
🕐 ZONING PERMITS - Manual Excel Import
   Executed: 2026-02-17 15:43:16
⚠️  Excel file not found: https:/docs.google.com/spreadsheets/d/1S6dGYohCEjZqXdLSsUXtvoA3w9jwz9qB/edit?usp=drive_link&ouid=111924636441288322199&rtpof=true&sd=true

📥 MANUAL DOWNLOAD REQUIRED:
   1. Go to: https://aca-prod.accela.com/BERKELEY/Cap/CapHome.aspx
      → Select 'Planning' module
   2. Export 'Active Zoning Projects' to Excel
   3. Save as: inputs/Active_Zoning_Projects.xlsx
   4. Re-run this cell


## 5. Fetch Building Permits

Building permits are issued after zoning approval.

In [15]:
# Fetch building permits
print("Fetching Building Permits...")
print("="*60)

cell_timestamp("Building Permits")

timer_start()

df_building = load_permits_from_api(
    'building_permits',
    limit=50000,
    app_token=APP_TOKEN
)

if df_building is not None:
    print(f"\nShape: {df_building.shape}")
    print(f"\nColumns:")
    for col in df_building.columns:
        print(f"  - {col}")
   
    print(f"\nSample records:")
    display(df_building.head(3))
timer_end()


Fetching Building Permits...
🕐 Building Permits
   Executed: 2026-02-17 15:43:56
🕐 Started: 2026-02-17 15:43:56
Using app token: z1ZX3Y2j...
Fetching building_permits from Berkeley Open Data...
Error fetching data: 403 Client Error: Forbidden
🕐 Completed: 2026-02-17 15:43:57
⏱️  Duration: 0.33 seconds


In [17]:
# ============================================================================
# ALTERNATIVE: Import Building Permits from Berkeley Open Data CSV - Feb2026
# ============================================================================

print("📥 LOADING BUILDING PERMITS FROM CSV")
print("="*70)

cell_timestamp("ZONING PERMITS - Manual Excel Import")


# Path to manually downloaded CSV
csv_file = Path('inputs/building_permits_raw.csv')

if csv_file.exists():
    print(f"✅ Found CSV file: {csv_file}")

    timer_start()

    
    try:
        # Read CSV
        df_building_csv = pd.read_csv(csv_file)
        
        print(f"   Loaded {len(df_building_csv)} permit records")
        print(f"   Columns: {list(df_building_csv.columns)[:5]}...")
        
        # Filter to housing/residential
        if 'Permit Type' in df_building_csv.columns:
            housing_types = ['Building', 'Residential', 'New Construction']
            df_building = df_building_csv[
                df_building_csv['Permit Type'].str.contains('|'.join(housing_types), 
                                                            case=False, 
                                                            na=False)
            ]
            print(f"   Filtered to {len(df_building)} housing-related permits")
        else:
            df_building = df_building_csv
            
        # Show sample
        print("\n📊 Sample data:")
        print(df_building.head(3))
        
    except Exception as e:
        print(f"❌ Error reading CSV: {e}")
        df_building = None

    timer_end()

        
else:
    print(f"⚠️  CSV file not found: {csv_file}")
    print("\n📥 MANUAL DOWNLOAD REQUIRED:")
    print("   1. Go to: https://data.cityofberkeley.info/d/ydr8-5enu")
    print("   2. Click: Export → CSV")
    print("   3. Save as: inputs/building_permits_raw.csv")
    print("   4. Re-run this cell")
    df_building = None

print("="*70)

📥 LOADING BUILDING PERMITS FROM CSV
🕐 ZONING PERMITS - Manual Excel Import
   Executed: 2026-02-17 15:44:33
⚠️  CSV file not found: inputs/building_permits_raw.csv

📥 MANUAL DOWNLOAD REQUIRED:
   1. Go to: https://data.cityofberkeley.info/d/ydr8-5enu
   2. Click: Export → CSV
   3. Save as: inputs/building_permits_raw.csv
   4. Re-run this cell


## Summarize Data Collection--note that failures of Accela may be remedied by new Clariti contract, if City mandates 

In [18]:
# ============================================================================
# DATA COLLECTION SUMMARY - Feb2026
# ============================================================================

print("📊 DATA COLLECTION SUMMARY")
print("="*70)

summary = {
    'Business Licenses': ('API', df_business is not None and len(df_business) > 0),
    'Zoning Permits': ('Excel/API', df_zoning is not None and len(df_zoning) > 0),
    'Building Permits': ('CSV/API', df_building is not None and len(df_building) > 0)
}

for dataset, (method, success) in summary.items():
    status = "✅" if success else "❌"
    count = 0
    
    if dataset == 'Business Licenses' and df_business is not None:
        count = len(df_business)
    elif dataset == 'Zoning Permits' and df_zoning is not None:
        count = len(df_zoning)
    elif dataset == 'Building Permits' and df_building is not None:
        count = len(df_building)
    
    print(f"{status} {dataset:20} ({method:10}) - {count:,} records")

print("\n" + "="*70)

# What worked?
api_worked = df_business is not None and len(df_business) > 0
manual_needed = (df_zoning is None or len(df_zoning) == 0) or \
                (df_building is None or len(df_building) == 0)

if api_worked and not manual_needed:
    print("🎉 All data successfully loaded!")
elif api_worked and manual_needed:
    print("⚠️  API worked for business licenses")
    print("   Manual downloads needed for zoning/building permits")
    print("   This is expected - Berkeley blocks these endpoints")
elif not api_worked:
    print("❌ No data loaded via API")
    print("   All datasets require manual download")

print("="*70)

📊 DATA COLLECTION SUMMARY
✅ Business Licenses    (API       ) - 100 records
❌ Zoning Permits       (Excel/API ) - 0 records
❌ Building Permits     (CSV/API   ) - 0 records

⚠️  API worked for business licenses
   Manual downloads needed for zoning/building permits
   This is expected - Berkeley blocks these endpoints


## 6. Document Data Schemas

Examine and document the schema for each dataset.

In [19]:
def document_schema(df, name):
    """Document dataframe schema"""
    print(f"\n{'='*60}")
    print(f"SCHEMA: {name}")
    print(f"{'='*60}")
    print(f"Records: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print()
    
    for col in df.columns:
        dtype = df[col].dtype
        non_null = df[col].notna().sum()
        pct = 100 * non_null / len(df)
        sample = df[col].dropna().iloc[0] if non_null > 0 else 'N/A'
        if isinstance(sample, str) and len(sample) > 40:
            sample = sample[:40] + '...'
        print(f"{col}")
        print(f"  Type: {dtype}, Non-null: {pct:.0f}%")
        print(f"  Sample: {sample}")
        print()

# Document schemas
if df_zoning is not None:
    document_schema(df_zoning, 'Zoning Permits')

if df_building is not None:
    document_schema(df_building, 'Building Permits')

## 7. Export Data

Save fetched data to CSV files.

In [ ]:
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d')

if df_zoning is not None:
    print('✅ Zoning data available')
else:
    print('⚠️ No zoning data - API blocked (403)')
    print('Manual download: https://data.cityofberkeley.info/d/vkhm-tsvp')

if df_building is not None:
    print('✅ Building data available')
else:
    print('⚠️ No building data - API blocked (403)')
    print('Manual download: https://data.cityofberkeley.info/d/ydr8-5enu')


## 8. Load to Database (Optional)

Load data into SQLite for Datasette.

In [ ]:
# Save to database
DB_PATH = CONFIG['paths']['database']

if df_zoning is not None:
    save_to_database(df_zoning, 'zoning_permits', DB_PATH)

if df_building is not None:
    save_to_database(df_building, 'building_permits', DB_PATH)

print(f"\nData loaded to: {DB_PATH}")

---

## Summary

This notebook:
- Connected to Berkeley Open Data Portal
- Downloaded zoning and building permits
- Documented data schemas
- Exported to CSV and SQLite

**Next:** Run `A2_address_standardization.ipynb` to standardize addresses.